# LoReFT

Replicates the emoji-chat demo from **"ReFT: Representation Finetuning for Language Models"** ([arXiv:2404.03592](https://arxiv.org/abs/2404.03592)) on Qwen2.5-1.5B-Instruct, end to end in one notebook:

1. **Training** — a rank-4 LoReFT intervention on `hidden_states` at layer 8 is trained with native EasySteer training on ten instruction→emoji examples (steered at the last prompt token with response-token supervision, following the official demo) and saved to `./trained_adapter/`.
2. **Steering** — the trained intervention is applied at the last prompt position in vLLM and makes the model answer in emojis.

Uses the EasySteer v2 steering API (`SteeringSpec` / `VectorSpec` / `ApplySpec`).

**Execution note:** Outputs are cleared to avoid mixing historical and native runs. The companion `run.py` executes the same 200-epoch recipe, supports DDP training, and evaluates all ten examples. See `README.md` for commands.


## Train the intervention

In [ ]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

import torch
from easysteer.training import PROMPT_TEMPLATE, train

device = "cuda"
MODEL = os.environ.get("EASYSTEER_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")


In [ ]:
# Rank-4 LoReFT intervention on `hidden_states` at layer 8.
# Keep the experiment settings explicit; train owns model loading,
# data collation and the Transformers Trainer integration.
steering_settings = {
    "algorithm": "loreft",
    "layer": 8,
    "component": "hidden_states",
    "rank": 4,
}


In [ ]:
import json

prompt_no_input_template = PROMPT_TEMPLATE
with open("training_examples.json", encoding="utf-8") as f:
    training_examples = json.load(f)

# Steer the last prompt token; supervise response tokens, including EOS.
examples = [[e["instruction"], e["emoji"]] for e in training_examples]


In [ ]:
steering_model, tokenizer = train(
    MODEL,
    examples,
    **steering_settings,
    device=device,
    prompt_template=prompt_no_input_template,
    num_train_epochs=200.0,
    seed=42,
    output_dir="./training-run",
    per_device_train_batch_size=10,
    learning_rate=4e-3,
    logging_steps=40,
    report_to=[],
    save_strategy="no",
    disable_tqdm=True,
    save_dir="./trained_adapter",
)


In [ ]:
import gc

# Release the HF training model before booting the vLLM engine. Moving
# the base model and intervention to CPU also releases GPU storage if
# an interactive notebook still holds another reference.
steering_model.to("cpu")
del steering_model
gc.collect()
torch.cuda.empty_cache()


## Steering

In [ ]:
from vllm import LLM, SamplingParams
from easysteer.training import load_checkpoint

# The checkpoint preserves the trained component, layer and prompt selection.
steering = load_checkpoint("./trained_adapter").to_spec()

# The trained rank-4 payload lets auto select the graph mode from its
# actual rank. Baseline requests below explicitly disable this default.
llm = LLM(
    model=MODEL,
    enable_steer_vector=True,
    steering_config=steering.model_dump_json(),
    # Headroom for the training process's residual CUDA context, which
    # shares the GPU with the engine in this single-notebook flow.
    gpu_memory_utilization=0.85,
)

In [ ]:
prompts = [
    prompt_no_input_template % "Who are you?",
    prompt_no_input_template % "Who am I?",
]
params = SamplingParams(temperature=0, max_tokens=16, skip_special_tokens=False)

baseline = llm.generate(prompts, params, use_tqdm=False, steering=False)
print("=====Baseline=====")
for out in baseline:
    print(out.outputs[0].text)

In [ ]:
steered = llm.generate(prompts, params, steering=steering, use_tqdm=False)
print("=====LoReFT Steered=====")
for out in steered:
    print(out.outputs[0].text)